# Session 2: Linear Programming for Power Control in Wireless Networks

**Course:** Optimization and Machine Learning — FH Aachen, SS 2026  
**Date:** 04.05.2026  
**Group 7**

---

## Problem Setting

A wireless network with $n = 4$ transmitter-receiver pairs. Each receiver $i$ requires a minimum **Signal-to-Interference-plus-Noise Ratio (SINR)**:

$$\text{SINR}_i = \frac{G_{ii} p_i}{\sum_{j \neq i} G_{ij} p_j + \sigma_i} \geq \gamma_i$$

Multiplying out the denominator linearises this constraint:

$$C\mathbf{p} \geq \boldsymbol{\gamma} \odot \boldsymbol{\sigma}, \qquad C_{ij} = \begin{cases} G_{ii} & i = j \\ -\gamma_i G_{ij} & i \neq j \end{cases}$$

In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt

## System Parameters

In [ ]:
G = np.array([
    [1.0, 0.1, 0.2, 0.1],
    [0.2, 1.2, 0.1, 0.2],
    [0.1, 0.3, 1.1, 0.2],
    [0.2, 0.1, 0.2, 1.0]
])
sigma = np.array([0.10, 0.10, 0.15, 0.10])
gamma = np.array([1.2, 1.0, 1.1, 0.9])
n = 4

## Task 1: Model Construction

In [ ]:
G_diag = np.diag(np.diag(G))
G_off  = G - G_diag
C   = G_diag - np.diag(gamma) @ G_off
rhs = gamma * sigma

print("Constraint matrix C:")
print(np.round(C, 3))
print("\nRHS (γ ⊙ σ):", np.round(rhs, 4))

# Plausibility: diagonal entries should be positive (direct signal),
# off-diagonal entries should be negative (interference penalty)
assert np.all(np.diag(C) > 0), "Diagonal must be positive"
print("\nPlausibility check passed: diagonal > 0 ✓")

## Task 2: Minimum Total Power

$$\min_{\mathbf{p} \geq 0} \sum_{i=1}^{4} p_i \quad \text{s.t.} \quad C\mathbf{p} \geq \boldsymbol{\gamma} \odot \boldsymbol{\sigma}$$

In [ ]:
p = cp.Variable(n)
prob = cp.Problem(cp.Minimize(cp.sum(p)), [C @ p >= rhs, p >= 0])
prob.solve()

p_opt = p.value
print(f"Status:              {prob.status}")
print(f"Minimum total power: {prob.value:.4f}")
print(f"Optimal powers p*:   {np.round(p_opt, 4)}")

# Verify SINR constraints
print("\nSINR verification:")
print(f"{'User':>6} {'SINR':>8} {'Required':>10} {'Satisfied':>11}")
for i in range(n):
    intf = sum(G[i,j] * p_opt[j] for j in range(n) if j != i)
    sinr = G[i,i] * p_opt[i] / (intf + sigma[i])
    ok = sinr >= gamma[i] - 1e-5
    print(f"{i+1:>6} {sinr:>8.4f} {gamma[i]:>10.1f} {'✓' if ok else '✗':>11}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar([f'User {i+1}' for i in range(n)], p_opt, color='steelblue', edgecolor='black')
ax.set_xlabel('Transmitter')
ax.set_ylabel('Transmit Power $p_i$')
ax.set_title(f'Optimal Power Allocation (Total = {prob.value:.3f})')
for bar, val in zip(bars, p_opt):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

## Task 3: Power Limits and Feasibility

Adding an upper bound $0 \leq p_i \leq p_{\max}$ for all $i$.  
**Parameter choice:** $p_{\max}$ swept from 0.01 to 1.0 in 100 steps — chosen to cover the range from clearly infeasible to clearly feasible, based on the unconstrained optimum.

In [ ]:
p_max_values = np.linspace(0.01, 1.0, 100)
feasible = []

for p_max in p_max_values:
    p_var = cp.Variable(n)
    prob_lim = cp.Problem(
        cp.Minimize(cp.sum(p_var)),
        [C @ p_var >= rhs, p_var >= 0, p_var <= p_max]
    )
    prob_lim.solve(solver=cp.CLARABEL)
    feasible.append(prob_lim.status == 'optimal')

feasible = np.array(feasible)
p_min_feasible = p_max_values[np.argmax(feasible)]
print(f"Smallest feasible p_max: {p_min_feasible:.3f}")
print(f"(Unconstrained optimum max component: {p_opt.max():.3f})")

# Report one feasible and one infeasible case
print(f"\nInfeasible example: p_max = {p_max_values[10]:.3f}")
print(f"Feasible example:   p_max = {p_max_values[-1]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p_max_values, feasible.astype(int), 'b-', linewidth=2, label='Feasible')
ax.axvline(p_min_feasible, color='red', linestyle='--',
           label=f'$p_{{\\max}}^{{\\min}}$ ≈ {p_min_feasible:.3f}')
ax.set_xlabel('$p_{\\max}$ (power limit per transmitter)')
ax.set_ylabel('Feasible (1 = yes, 0 = no)')
ax.set_title('Feasibility as a Function of $p_{\\max}$')
ax.set_yticks([0, 1])
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## Task 4: Uniform QoS Scaling

Replace $\boldsymbol{\gamma}$ by $\alpha\boldsymbol{\gamma}$. Find the largest $\alpha$ such that the problem remains feasible with $p_i \leq 2$.  
**Method:** Parameter sweep over $\alpha \in [0.1, 5.0]$ in 200 steps — fine enough to locate the transition clearly.

In [ ]:
alpha_values = np.linspace(0.1, 5.0, 200)
p_max_qos = 2.0
feasible_alpha = []
total_power_alpha = []

for alpha in alpha_values:
    gamma_a = alpha * gamma
    C_a = G_diag - np.diag(gamma_a) @ G_off
    rhs_a = gamma_a * sigma
    p_var = cp.Variable(n)
    prob_a = cp.Problem(
        cp.Minimize(cp.sum(p_var)),
        [C_a @ p_var >= rhs_a, p_var >= 0, p_var <= p_max_qos]
    )
    prob_a.solve(solver=cp.CLARABEL)
    ok = prob_a.status == 'optimal'
    feasible_alpha.append(ok)
    total_power_alpha.append(prob_a.value if ok else np.nan)

feasible_alpha = np.array(feasible_alpha)
alpha_max = alpha_values[np.where(feasible_alpha)[0][-1]]
print(f"Maximum feasible α (with p_i ≤ {p_max_qos}): {alpha_max:.3f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(alpha_values, feasible_alpha.astype(int), 'b-', linewidth=2)
ax1.axvline(alpha_max, color='red', linestyle='--', label=f'$\\alpha_{{max}}$ ≈ {alpha_max:.2f}')
ax1.set_xlabel('$\\alpha$ (QoS scaling factor)')
ax1.set_ylabel('Feasible')
ax1.set_title('Feasibility vs. $\\alpha$')
ax1.set_yticks([0, 1])
ax1.legend()
ax1.grid(True, alpha=0.4)

ax2.plot(alpha_values, total_power_alpha, 'g-', linewidth=2)
ax2.axvline(alpha_max, color='red', linestyle='--', label=f'$\\alpha_{{max}}$ ≈ {alpha_max:.2f}')
ax2.set_xlabel('$\\alpha$')
ax2.set_ylabel('Minimum total power')
ax2.set_title('Total Power vs. $\\alpha$ (feasible region)')
ax2.legend()
ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

## Task 5: Random Instance Generator

In [ ]:
def generate_instance(n, seed=None):
    """Random wireless network: strong direct gains, weaker interference."""
    rng = np.random.default_rng(seed)
    G_r = rng.uniform(0.05, 0.3, (n, n))
    np.fill_diagonal(G_r, rng.uniform(0.8, 1.5, n))
    return G_r, rng.uniform(0.05, 0.2, n), rng.uniform(0.5, 1.5, n)


def solve_power_control(G_r, sigma_r, gamma_r, p_max=None):
    n_r = len(gamma_r)
    G_d = np.diag(np.diag(G_r))
    C_r = G_d - np.diag(gamma_r) @ (G_r - G_d)
    rhs_r = gamma_r * sigma_r
    p_v = cp.Variable(n_r)
    cons = [C_r @ p_v >= rhs_r, p_v >= 0]
    if p_max is not None:
        cons.append(p_v <= p_max)
    prob = cp.Problem(cp.Minimize(cp.sum(p_v)), cons)
    prob.solve(solver=cp.CLARABEL)
    active = int(np.sum(p_v.value >= p_max - 1e-4)) if prob.status == 'optimal' and p_max else 0
    return prob.status, prob.value, active


p_max_test = 2.0
print(f"{'Instance':>10} {'Status':>14} {'Total Power':>13} {'Active Limits':>15}")
print("-" * 56)
for seed in range(5):
    G_r, sigma_r, gamma_r = generate_instance(n=4, seed=seed)
    status, val, active = solve_power_control(G_r, sigma_r, gamma_r, p_max=p_max_test)
    if status == 'optimal':
        print(f"{seed+1:>10} {'feasible':>14} {val:>13.4f} {active:>15}")
    else:
        print(f"{seed+1:>10} {'infeasible':>14} {'—':>13} {'—':>15}")